In [1]:
# 08_rag_demo.ipynb
# -------------------------------------------------------
# RAG demo: retrieve top-k and synthesize an answer
# -------------------------------------------------------

import os
import yaml
import json
import numpy as np
import pandas as pd
import faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer
from openai import OpenAI
from dotenv import load_dotenv

# -------- Config (edit if your paths differ) --------
with open("config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

DATA_DIR = Path(config.get("paths", {}).get("data", "data"))
EMB_DIR = Path(config.get("paths", {}).get("embeddings", "embeddings"))
INDEX_DIR = Path(config.get("paths", {}).get("indices", "indices"))

MODEL_ST = config.get("models", {}).get(
    "sentence_transformer", "all-MiniLM-L6-v2"
)

OPENAI_EMB_MODEL = config.get("models", {}).get(
    "openai_embedding", "text-embedding-3-small"
)

OPENAI_CHAT_MODEL = config.get("models", {}).get(
    "openai_chat", "gpt-4o-mini"
)

TOP_K = config.get("retrieval", {}).get("top_k", 5)

print("Top-K:", TOP_K)


# -------- Load index, metadata, and chunks --------
META_JSON = EMB_DIR / "metadata.json"
META_JSONL = EMB_DIR / "metadata.jsonl"
META_PARQUET = EMB_DIR / "metadata.parquet"

if META_PARQUET.exists():
    metadata_df = pd.read_parquet(META_PARQUET)
    print("Loaded metadata from Parquet")
elif META_JSON.exists():
    metadata_df = pd.read_json(META_JSON)
    print("Loaded metadata from JSON")
elif META_JSONL.exists():
    metadata_df = pd.read_json(META_JSONL, lines=True)
    print("Loaded metadata from JSONL")
else:
    raise FileNotFoundError("No metadata file found")

print("Metadata records:", len(metadata_df))
display(metadata_df.head())

# -------- Load FAISS index --------
index_st = faiss.read_index(str(INDEX_DIR / "faiss_index_st.index"))
index_oa = faiss.read_index(str(INDEX_DIR / "faiss_index_openai.index"))

print("ST vectors:", index_st.ntotal)
print("OpenAI vectors:", index_oa.ntotal)

# --------Load models----------
# SentenceTransformer
st_model = SentenceTransformer(MODEL_ST)

# OpenAI
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

if api_key:
    client = OpenAI(api_key=api_key)
    OPENAI_ENABLED = True
    print("✅ OpenAI enabled")
else:
    client = None
    OPENAI_ENABLED = False
    print("⚠️ OpenAI disabled (no API key)")


# -------- Embedding function --------
def embed_query_st(query: str):
    vec = st_model.encode([query], convert_to_numpy=True)
    return vec.astype("float32")


def embed_query_openai(query: str):
    if not OPENAI_ENABLED:
        raise RuntimeError("OpenAI is not enabled")

    resp = client.embeddings.create(
        model=OPENAI_EMB_MODEL,
        input=query
    )
    vec = np.array(resp.data[0].embedding, dtype="float32")
    return vec.reshape(1, -1)


# -------- Retrieval function --------
def retrieve_context(query, model_type="st", top_k=5):
    if model_type == "st":
        query_vec = embed_query_st(query)
        index = index_st
    elif model_type == "openai":
        if not OPENAI_ENABLED:
            print("⚠️ OpenAI retrieval skipped")
            return []
        query_vec = embed_query_openai(query)
        index = index_oa
    else:
        raise ValueError("Invalid model_type")

    distances, indices = index.search(query_vec, top_k)

    contexts = []
    for idx in indices[0]:
        row = metadata_df.iloc[idx]
        contexts.append(row.get("text", ""))

    return contexts


#--------- prompt template --------
def build_prompt(query, contexts):
    context_block = "\n\n".join(contexts)

    prompt = f"""
ඔබ ආයුර්වේද වෛද්‍ය සහායකයෙකි.

පහත Neo4j graph මගින් ලබාගත් සන්දර්භය (Context) භාවිතා කර
ප්‍රශ්නයට **සිංහල භාෂාවෙන්** පිළිතුරු දෙන්න.

Context:
{context_block}

Question:
{query}

Instructions:
- සිංහලෙන් පිළිතුරු දෙන්න
- වෛද්‍යමය ලෙස නිවැරදි විය යුතුය
- කෙටි හා පැහැදිලි විය යුතුය
- ලබාදුන් සන්දර්භය අනුව රෝගයේ නාමය පැහැදිලිව සඳහන් කරන්න
- රෝගය හඳුනාගත නොහැකි නම් “නියමිතව හඳුනාගත නොහැක” ලෙස සඳහන් කරන්න

Answer:
"""
    return prompt.strip()

# --------Generate Answer (RAG)--------
def generate_answer(query, model_type="st"):
    contexts = retrieve_context(query, model_type=model_type, top_k=TOP_K)

    if not contexts:
        return "No relevant context retrieved."

    prompt = build_prompt(query, contexts)

    if not OPENAI_ENABLED:
        return "OpenAI is disabled. Retrieved context:\n\n" + "\n".join(contexts[:2])

    response = client.chat.completions.create(
        model=OPENAI_CHAT_MODEL,
        messages=[
            {"role": "system", "content": "You are a knowledgeable Ayurvedic assistant."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.2
    )

    return response.choices[0].message.content


# -------- English RAG Demo --------
query = "Oily skin with inflamed pimples and blackheads"

answer = generate_answer(query, model_type="openai")
print("🔎 Query:\n", query)
print("\n🧠 RAG Answer:\n")
print(answer)


# -------- Sinhala RAG Demo --------
query_si = "බෙල්ලේ තද බවක් දැනීමට, බඩ පුරවා දැමීම, නිදා ගැනීමේ අපහසුතාව කුමන රෝගයක් නිසා ඇති විය හැකිද?"

answer_si = generate_answer(query_si, model_type="openai")
print("🔎 Sinhala Query:\n", query_si)
print("\n🧠 RAG Answer:\n")
print(answer_si)


# -------- SentenceTransformer-Only RAG (Offline Mode) --------
query = "Chronic itching with dry, cracked skin"

answer = generate_answer(query, model_type="st")
print("🔎 Query:\n", query)
print("\n🧠 Retrieved Context (Offline Mode):\n")
print(answer)


c:\Users\ASUS\anaconda3\envs\genai_research\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Top-K: 5
Loaded metadata from JSON
Metadata records: 20


,chunk_id,lang,disease_name_en,disease_name_si
0,D001,en,Tarunya Pitika (Acne vulgaris),තරුණ්‍ය පිටිකා (මුඛදූෂිකා)
1,D001,si,Tarunya Pitika (Acne vulgaris),තරුණ්‍ය පිටිකා (මුඛදූෂිකා)
2,D002,en,Vicharchika (Eczema),විචර්චිකා (ඇක්සීමා)
3,D002,si,Vicharchika (Eczema),විචර්චිකා (ඇක්සීමා)
4,D003,en,Mashaka (Elevated Mole),මෂක (ඉහළට නෙරා ඇති ලපය)


ST vectors: 20
OpenAI vectors: 20
✅ OpenAI enabled
🔎 Query:
 Oily skin with inflamed pimples and blackheads

🧠 RAG Answer:

මෙම ලක්ෂණ මත පදනම්ව, ඔබට "අක්නේ" (Acne) යන රෝගය ඇති බවක් පෙනේ. මේ සඳහා හේතු වශයෙන් තෙත් සම, ආහාර පද්ධතිය, හෝ හර්මෝනල් වෙනස්කම් වැනි කරුණු සම්බන්ධ වේ. 

ආයුර්වේදය අනුව, මේ රෝගය "පිතා" (Pitta) දෝෂය වැඩිවීමෙන් සිදුවන බවට සැලකේ. 

ඉතාමත් වැදගත් වන්නේ, නිවැරදි ආහාරය, ජලය පවා, සහ සමේ සෞඛ්‍යය රැක ගැනීමයි. 

ඉතාමත් ප්‍රයෝජනවත් වන සෞඛ්‍ය උපදෙස්:
1. තෙල් සහිත ආහාර වලින් වළකිනු.
2. පළතුරු සහ එළවළු බොහෝමයක් භාවිතා කරන්න.
3. නිසි ලෙස ජලය පාන කරන්න.
4. සම පිරිසිදු කිරීම සඳහා නිවැරදි සම්පත් භාවිතා කරන්න.

ඔබට මෙම ලක්ෂණ සහිත නම්, වෛද්‍යවරයෙකුට පරීක්ෂාවක් සඳහා යාමට යෝජනා කරනවා.
🔎 Sinhala Query:
 බෙල්ලේ තද බවක් දැනීමට, බඩ පුරවා දැමීම, නිදා ගැනීමේ අපහසුතාව කුමන රෝගයක් නිසා ඇති විය හැකිද?

🧠 RAG Answer:

මෙම ලක්ෂණ තුන (බෙල්ලේ තද බව, බඩ පුරවා දැමීම, නිදා ගැනීමේ අපහසුතාව) සම්බන්ධ විය හැකි රෝගයක් ලෙස "ආහාර පෝෂණය හා පචනයේ ගැටළු" (Functional Dyspepsia) හෝ "ගැස්ට්‍රයිටිස්" (Gastritis) යන ර